# Lab 26 — Optimize ClaimsIQ Using Redis Caching

**Module:** Performance Optimization · Day 15 · Session 01
**Duration:** ~45-55 minutes

### What you will do
Add a cache-aside layer in front of `get_customer_profile`, measure the
real latency difference between a cache miss and a cache hit, and test
both TTL expiry and manual invalidation.

### Prerequisite
ClaimsIQ Notebooks 00-01 must have run already, so
`mcp_snowflake_server.py` exists on disk.

### Before you start — install and run Redis locally

```bash
# macOS
brew install redis && brew services start redis

# Ubuntu/Debian
sudo apt-get install redis-server && sudo service redis-server start

# Or via Docker, on any OS
docker run -d -p 6379:6379 redis
```

## Step 1 — Install & connect

In [1]:
%pip install -q redis snowflake-connector-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time
import redis
from mcp_snowflake_server import claims_server, SimpleMCPClient

mcp_client = SimpleMCPClient(claims_server)
mcp_client.connect()

redis_client = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
redis_client.ping()
print("Redis connected.")

Connected to 'claimsiq-snowflake'. Discovered 6 tools.


ConnectionError: Error 10061 connecting to localhost:6379. No connection could be made because the target machine actively refused it.

## Step 2 — Write `get_customer_profile_cached()`

Cache-aside pattern: check Redis first, fall back to the real MCP tool
call on a miss, then populate the cache.

In [3]:
CACHE_TTL_SECONDS = 300  # 5 minutes

def get_customer_profile_cached(customer_id: str, verbose=True) -> dict:
    cache_key = f"profile:{customer_id}"
    cached = redis_client.get(cache_key)
    if cached:
        if verbose:
            print(f"[CACHE HIT] {cache_key}")
        return json.loads(cached)

    if verbose:
        print(f"[CACHE MISS] {cache_key} — calling Snowflake via MCP")
    result = mcp_client.call_tool("get_customer_profile", customer_id=customer_id)
    redis_client.setex(cache_key, CACHE_TTL_SECONDS, json.dumps(result, default=str))
    return result

print("get_customer_profile_cached() ready.")

get_customer_profile_cached() ready.


## Step 3 — Clear any leftover cache, then time an uncached call (cache miss)

In [4]:
redis_client.delete("profile:CUST99001")  # ensure a clean miss

start = time.time()
result = get_customer_profile_cached("CUST99001")
miss_latency = time.time() - start
print(f"\nCache MISS latency: {miss_latency*1000:.1f}ms")

ConnectionError: Error 10061 connecting to localhost:6379. No connection could be made because the target machine actively refused it.

## Step 4 — Time the SAME call again (cache hit)

In [ ]:
start = time.time()
result = get_customer_profile_cached("CUST99001")
hit_latency = time.time() - start
print(f"Cache HIT latency: {hit_latency*1000:.1f}ms")

## Step 5 — Compare the latency difference

In [ ]:
speedup = miss_latency / hit_latency if hit_latency > 0 else float("inf")
print(f"Cache miss: {miss_latency*1000:.1f}ms")
print(f"Cache hit:  {hit_latency*1000:.1f}ms")
print(f"Speedup:    {speedup:.1f}x faster on a cache hit")

## Step 6 — Set a short TTL, confirm expiry works

Use a much shorter TTL for this test so you don't have to wait 5
minutes to see it expire.

In [ ]:
redis_client.delete("profile:CUST99001")
redis_client.setex("profile:CUST99001", 3, json.dumps(mcp_client.call_tool("get_customer_profile", customer_id="CUST99001"), default=str))
print("Cached with a 3-second TTL.")

print("Immediately after caching:", redis_client.get("profile:CUST99001") is not None)
time.sleep(4)
print("After waiting 4 seconds:  ", redis_client.get("profile:CUST99001") is not None)

## Step 7 — Manually invalidate on a simulated data change

Event-based invalidation: if a new fraud signal were written for this
customer, the STALE cached profile should be cleared immediately,
rather than waiting for the TTL to expire naturally.

In [ ]:
# Populate the cache again
get_customer_profile_cached("CUST99001", verbose=False)
print("Cached before 'data change':", redis_client.get("profile:CUST99001") is not None)

# Simulate: a new fraud signal was just written for this customer elsewhere in the system
def invalidate_customer_cache(customer_id: str):
    redis_client.delete(f"profile:{customer_id}")
    print(f"Invalidated cache for {customer_id} due to a data change.")

invalidate_customer_cache("CUST99001")
print("Cached after invalidation:  ", redis_client.get("profile:CUST99001") is not None)

## Deliverable

1. The miss/hit latency numbers from Steps 3-5.
2. Confirmation from Step 6 that the TTL expiry worked as expected.
3. One paragraph: `get_customer_profile` was chosen for caching in this
   lab because it's read frequently and changes rarely. Would
   `check_fraud_signals` be an equally good caching candidate? Referring
   back to Session 01's slides, justify your answer with the specific
   trade-off involved.